# Sheet 0.5: Optimization

**Roadmap topic 0.5.** Read first: *Why Momentum Really Works* (Distill), MML ch. 7, Ruder's overview, the Adam and AdamW papers.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. Your optimizers may not call anything from `torch.optim` except the `Optimizer` base class. CPU is fine; Exercise 4's experiment is faster on a Colab GPU.

In [ ]:
import math, torch
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

## Exercise 1: Conditioning sets the speed of gradient descent

For `f(x) = ½ xᵀAx` with eigenvalues in `[μ, L]` and `κ = L/μ`:
- `gd_optimal_lr(mu, L)`: the step size that minimises the worst-case contraction factor, and `gd_rate(mu, L)`: that factor.
- `heavy_ball_params(mu, L)`: optimal `(lr, beta)` for Polyak heavy-ball momentum, and `heavy_ball_rate(mu, L)`.
- `run(A, x0, lr, beta, steps)`: runs `x ← x - lr·∇f(x) + beta·(x - x_prev)` and returns the list of `‖x_t‖`.

The check measures the empirical contraction and compares with your formulas for κ = 10, 100, 1000.

In [ ]:
def gd_optimal_lr(mu, L): raise NotImplementedError
def gd_rate(mu, L): raise NotImplementedError
def heavy_ball_params(mu, L): raise NotImplementedError
def heavy_ball_rate(mu, L): raise NotImplementedError
def run(A, x0, lr, beta, steps):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    n = 50
    Q, _ = torch.linalg.qr(torch.randn(n, n))
    for kappa in (10, 100, 1000):
        mu, L = 1.0, float(kappa)
        A = (Q * torch.linspace(mu, L, n)) @ Q.T
        x0 = Q @ torch.ones(n)
        T = 60 * int(math.sqrt(kappa)) + 200
        def measured(norms, T): return (norms[-1] / norms[-T // 2]) ** (1 / (T - T // 2))
        r_gd = measured(run(A, x0, gd_optimal_lr(mu, L), 0.0, T), T)
        assert abs(r_gd - gd_rate(mu, L)) < 0.02 * (1 - gd_rate(mu, L)) + 1e-3, f"kappa={kappa}: GD rate {r_gd:.5f} vs predicted {gd_rate(mu, L):.5f}"
        lr, beta = heavy_ball_params(mu, L)
        r_hb = measured(run(A, x0, lr, beta, T), T)
        assert r_hb < heavy_ball_rate(mu, L) + 0.02, f"kappa={kappa}: heavy-ball {r_hb:.4f} vs predicted {heavy_ball_rate(mu, L):.4f}"
        assert r_hb < r_gd, "momentum should beat GD"
        print(f"kappa={kappa:5d}  GD {r_gd:.4f} (pred {gd_rate(mu,L):.4f})   HB {r_hb:.4f} (pred {heavy_ball_rate(mu,L):.4f})")
    print("PASS")
_check1()

**Written (1a):** How many iterations does GD need to reduce the error by 10⁶ at κ = 1000? And heavy-ball? What does this suggest about preconditioning (which is what Adam approximates)?

**Your answer:**

## Exercise 2: Optimizers from scratch

Implement as subclasses of `torch.optim.Optimizer`, matching the PyTorch update rules and defaults exactly:
- `MySGD(params, lr, momentum=0, nesterov=False, weight_decay=0)`
- `MyRMSprop(params, lr, alpha=0.99, eps=1e-8)`
- `MyAdam(params, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`: L2-style weight decay added to the gradient, as in `torch.optim.Adam`
- `MyAdamW(params, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)`: decoupled decay

Read the pseudo-code in each class's PyTorch documentation page; the details (where `eps` goes, when bias correction applies) are the exercise.

In [ ]:
from torch.optim import Optimizer
class MySGD(Optimizer):
    def __init__(self, params, lr, momentum=0.0, nesterov=False, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self):
        # YOUR CODE
        raise NotImplementedError
class MyRMSprop(Optimizer):
    def __init__(self, params, lr, alpha=0.99, eps=1e-8):
        super().__init__(params, dict(lr=lr, alpha=alpha, eps=eps))
    @torch.no_grad()
    def step(self):
        # YOUR CODE
        raise NotImplementedError
class MyAdam(Optimizer):
    def __init__(self, params, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self):
        # YOUR CODE
        raise NotImplementedError
class MyAdamW(Optimizer):
    def __init__(self, params, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self):
        # YOUR CODE
        raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    def traj(make_opt):
        torch.manual_seed(1)
        net = torch.nn.Sequential(torch.nn.Linear(10, 32), torch.nn.Tanh(), torch.nn.Linear(32, 3))
        X = torch.randn(64, 10); y = torch.randint(0, 3, (64,)); opt = make_opt(net.parameters())
        for _ in range(100):
            opt.zero_grad(); torch.nn.functional.cross_entropy(net(X), y).backward(); opt.step()
        return torch.cat([p.detach().flatten() for p in net.parameters()])
    O = torch.optim
    cases = {
        "SGD": (lambda p: MySGD(p, .1), lambda p: O.SGD(p, .1)),
        "SGD momentum+wd": (lambda p: MySGD(p, .05, .9, weight_decay=1e-2), lambda p: O.SGD(p, .05, .9, weight_decay=1e-2)),
        "SGD nesterov": (lambda p: MySGD(p, .05, .9, nesterov=True), lambda p: O.SGD(p, .05, .9, nesterov=True)),
        "RMSprop": (lambda p: MyRMSprop(p, 1e-3), lambda p: O.RMSprop(p, 1e-3)),
        "Adam+L2": (lambda p: MyAdam(p, 1e-2, weight_decay=1e-2), lambda p: O.Adam(p, 1e-2, weight_decay=1e-2)),
        "AdamW": (lambda p: MyAdamW(p, 1e-2, weight_decay=0.1), lambda p: O.AdamW(p, 1e-2, weight_decay=0.1)),
    }
    for name, (mine, ref) in cases.items():
        a, b = traj(mine), traj(ref)
        assert torch.allclose(a, b, atol=1e-7), f"{name}: max param diff after 100 steps {(a-b).abs().max():.2e}"
    print("PASS  all 6 optimizers match torch.optim for 100 steps")
_check2()

## Exercise 3: Why AdamW is not Adam + L2

Take a single weight `w` whose loss gradient is **zero** (only weight decay acts on it, like a parameter of an unused feature). Starting from `w = 1.0`, run 200 steps with `lr = 1e-3`:
- `adam_l2_trajectory(wd)` using `MyAdam(..., weight_decay=wd)`
- `adamw_trajectory(wd)` using `MyAdamW(..., weight_decay=wd)`

Return the list of `w` values. Set `w.grad = torch.zeros(1)` each step so the optimizer sees zero loss gradient.

Then predict the shapes **before** running: one is exponential decay that depends on `wd`; the other is nearly linear and barely depends on `wd`. Which is which, and why?

In [ ]:
def adam_l2_trajectory(wd, steps=200, lr=1e-3):
    # YOUR CODE
    raise NotImplementedError
def adamw_trajectory(wd, steps=200, lr=1e-3):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    for wd in (0.01, 0.1, 1.0):
        aw = adamw_trajectory(wd)
        assert abs(aw[-1] - (1 - 1e-3 * wd) ** 200) < 1e-9, f"AdamW wd={wd}: expected exponential decay"
    l2 = [adam_l2_trajectory(wd)[-1] for wd in (0.01, 0.1, 1.0)]
    assert all(abs(x - (1 - 200 * 1e-3)) < 0.01 for x in l2), f"Adam+L2 should move about lr per step regardless of wd: {l2}"
    print(f"PASS  after 200 steps  Adam+L2: {[round(x,4) for x in l2]}   AdamW: {[round(adamw_trajectory(wd)[-1],4) for wd in (0.01,0.1,1.0)]}")
_check3()

**Written (3a):** Explain the result. Then explain what goes wrong with Adam + L2 on a weight that *does* have large, noisy gradients: is it regularised more or less than a weight with small gradients?

**Your answer:**

### On Colab: the real-data version
Train a small CNN on CIFAR-10 for 10 epochs with `MyAdam(weight_decay=5e-4)` vs `MyAdamW(weight_decay=0.05)`, 3 seeds each, and plot test accuracy and the weight norm per layer. Write 3 lines on what you see.

In [ ]:
# Colab: your CIFAR-10 experiment here (torchvision.datasets.CIFAR10, a 4-layer CNN, batch 256)

## Exercise 4: Sharpness and the edge of stability

`sharpness(loss_fn, params, iters)` estimates the largest Hessian eigenvalue of the loss w.r.t. a list of parameter tensors using **power iteration on Hessian-vector products** (double backward with `torch.autograd.grad(..., create_graph=True)`). Never build the Hessian.

In [ ]:
def sharpness(loss_fn, params, iters=100):
    # loss_fn() returns a scalar loss; params is a list of tensors with requires_grad
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    torch.manual_seed(3)
    X = torch.randn(40, 3); y = torch.randn(40, 1)
    W1 = torch.randn(3, 4, requires_grad=True); W2 = torch.randn(4, 1, requires_grad=True)
    f = lambda: ((torch.tanh(X @ W1) @ W2 - y) ** 2).mean()
    est = sharpness(f, [W1, W2], iters=300)
    flat = torch.cat([W1.detach().flatten(), W2.detach().flatten()])
    def f_flat(z): return ((torch.tanh(X @ z[:12].view(3, 4)) @ z[12:].view(4, 1) - y) ** 2).mean()
    H = torch.autograd.functional.hessian(f_flat, flat)
    ev = torch.linalg.eigvalsh(H)
    top = ev[ev.abs().argmax()]
    assert abs(est - top) / abs(top) < 1e-4, f"sharpness {est:.5f} vs true top eigenvalue {top:.5f}"
    print(f"PASS  sharpness {est:.4f}")
_check4()

### Experiment (CPU ok, GPU faster): watch the edge of stability

Train a 3-layer tanh MLP on 1,000 CIFAR-10 images (flattened, 10 classes) with **full-batch** gradient descent at `lr = 0.02` and then `lr = 0.04`. Every 25 steps record the loss and `sharpness(...)`. Plot sharpness with a horizontal line at `2/lr`.

You should see sharpness rise ("progressive sharpening") until it reaches about `2/lr`, then hover there while the loss keeps falling non-monotonically. Compare with Cohen et al. (2021), figure 1.

In [ ]:
# YOUR experiment code here

**Written (4a):** Classical theory says GD diverges when sharpness exceeds 2/lr. Your plot shows it hovering there without diverging. In 3-4 sentences, what does this tell you about why learning-rate warmup helps large models?

## Checkpoint

Explain to a colleague, without notes: why Adam needs bias correction (what goes wrong at step 1 without it), why warmup helps, and what happens to the *effective* weight decay in AdamW when you halve the learning rate. Check yourself against Exercise 3 and the AdamW paper section 2.

---
### Hints for self-grading
- 1a: GD needs about ln(10⁶)/ln(1/0.998) ≈ 6,900 steps at κ = 1000; heavy-ball about ln(10⁶)/ln(1/0.939) ≈ 220. Preconditioning shrinks κ itself.
- 3a: Adam normalises the L2 gradient `wd·w` by its own RMS, so the step is about `lr·sign(w)` whatever wd is. With large noisy loss gradients, `v` is large and the decay term gets divided by it: those weights are regularised *less*. AdamW applies decay outside the normalisation, so every weight decays at rate `lr·wd`.
- 4a: early in training, sharpness can be far above 2/lr for the target lr, and the self-stabilisation needs time; warmup lets sharpness come down (or the model reach flatter regions) before the full lr is applied.